# Notebook 01 — IFC Graph Construction

Builds the typed IFC property graph G=(V,E,τ) from the reference IFC file.
Inspects node and edge statistics, visualises the graph, and saves it for reuse.

**No GPU required.**

In [ ]:
import os, sys, json
REPO = 'ifc-graphrag-dt'
if os.path.exists(REPO): !cd {REPO} && git pull --quiet
else: !git clone https://github.com/aiwithprashant/ifc-graphrag-dt.git
sys.path.insert(0, REPO)

import networkx as nx
import matplotlib.pyplot as plt
from collections import Counter
from pathlib import Path

from pipeline.layer1_retriever.ifc_graph_builder import IFCGraphBuilder

IFC_PATH  = f'{REPO}/benchmark/ifc_reference_models/duplex.ifc'
GRAPH_OUT = f'{REPO}/outputs/graphs/ifc_graph.json'
os.makedirs(f'{REPO}/outputs/graphs', exist_ok=True)
print('Imports OK')

In [ ]:
# ── Cell 2: Build graph ───────────────────────────────────────────────────────
import logging
logging.basicConfig(level=logging.INFO, format='%(levelname)s: %(message)s')

if Path(GRAPH_OUT).exists():
    print('Loading cached graph...')
    G = IFCGraphBuilder.load_graph(GRAPH_OUT)
else:
    print('Building IFC graph from file...')
    builder = IFCGraphBuilder(IFC_PATH)
    G = builder.build()
    builder.save_graph(GRAPH_OUT)

print(f'\nGraph: {G.number_of_nodes()} nodes, {G.number_of_edges()} edges')

In [ ]:
# ── Cell 3: Graph summary stats ───────────────────────────────────────────────
# Rebuild builder for summary (graph already saved)
builder = IFCGraphBuilder(IFC_PATH)
builder.model = None  # force reload only if needed
builder.G = G
builder._node_map = {data['global_id']: data for _, data in G.nodes(data=True) if 'global_id' in data}

# Manual summary
node_types = Counter(data.get('ifc_type','?') for _, data in G.nodes(data=True))
edge_types = Counter(data.get('relation_type','?') for _, _, data in G.edges(data=True))

print('TOP 15 ENTITY TYPES')
print(f'{"Type":<45} {"Count":>6}')
print('-' * 55)
for t, n in node_types.most_common(15):
    print(f'{t:<45} {n:>6}')

print('\nRELATION TYPE DISTRIBUTION')
print(f'{"Relation":<48} {"Count":>6}')
print('-' * 58)
for r, n in edge_types.most_common():
    print(f'{r:<48} {n:>6}')

In [ ]:
# ── Cell 4: Entity type distribution chart ────────────────────────────────────
top_n = 20
top_types = node_types.most_common(top_n)
labels = [t[0].replace('Ifc','') for t in top_types]
values = [t[1] for t in top_types]

plt.figure(figsize=(14, 5))
bars = plt.barh(labels[::-1], values[::-1], color='#2E5FA3')
plt.xlabel('Instance Count')
plt.title(f'Top {top_n} IFC Entity Types in Reference Model')
plt.tight_layout()
plt.savefig(f'{REPO}/outputs/figures/01_entity_type_distribution.png', dpi=150)
plt.show()

In [ ]:
# ── Cell 5: Degree distribution ───────────────────────────────────────────────
in_degrees  = [d for _, d in G.in_degree()]
out_degrees = [d for _, d in G.out_degree()]

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].hist(in_degrees,  bins=30, color='#1D9E75', edgecolor='white')
axes[0].set_title('In-degree Distribution')
axes[0].set_xlabel('In-degree (number of incoming relations)')
axes[0].set_ylabel('Node count')

axes[1].hist(out_degrees, bins=30, color='#D85A30', edgecolor='white')
axes[1].set_title('Out-degree Distribution')
axes[1].set_xlabel('Out-degree (number of outgoing relations)')

plt.tight_layout()
plt.savefig(f'{REPO}/outputs/figures/01_degree_distribution.png', dpi=150)
plt.show()
print(f'Max in-degree: {max(in_degrees)}  |  Max out-degree: {max(out_degrees)}')
print(f'Mean in-degree: {sum(in_degrees)/len(in_degrees):.2f}  |  Mean out-degree: {sum(out_degrees)/len(out_degrees):.2f}')

In [ ]:
# ── Cell 6: Extract a spatial containment subgraph for inspection ─────────────
from pipeline.layer1_retriever.khop_traversal import KHopTraversal

# Find all IfcSpace nodes
space_nodes = [n for n, d in G.nodes(data=True) if d.get('ifc_type') == 'IfcSpace']
print(f'Found {len(space_nodes)} IfcSpace nodes')

if space_nodes:
    # Traverse 2 hops from the first space
    traversal = KHopTraversal(G, max_depth=2, bidirectional=True)
    result = traversal.traverse([space_nodes[0]])
    print(f'\n2-hop traversal from first IfcSpace:')
    print(result.summary())

    entity_list = result.to_entity_list()
    print(f'\nEntities in subgraph:')
    type_counts = Counter(e['ifc_type'] for e in entity_list)
    for t, n in sorted(type_counts.items()):
        print(f'  {t}: {n}')

In [ ]:
# ── Cell 7: Verify tier-depth requirements ────────────────────────────────────
# Demonstrate the theoretical justification for k>=3 at Tier 3

print('Theoretical justification: path lengths to required Tier 3 subgraphs')
print('=' * 65)

if space_nodes and len(space_nodes) > 0:
    for depth in [1, 2, 3, 4]:
        t = KHopTraversal(G, max_depth=depth, bidirectional=True)
        r = t.traverse([space_nodes[0]])
        edge_types_found = set(r.relation_type_counts.keys())
        print(f'  depth={depth}: {r.node_count:>4} nodes, {r.edge_count:>4} edges | '
              f'relations: {len(edge_types_found)} types')

print('\nConclusion: Tier 3 system prompts require k>=3 to recover')
print('containment + port connectivity + system membership chains.')
print('\nGraph construction complete. Proceed to Notebook 02.')